In [1]:
import os
from pyspark.sql import SparkSession

assert os.environ.get("MINIO_ROOT_USER"), "Identifiant MinIO absent"
assert os.environ.get("MINIO_ROOT_PASSWORD"), "Mot de passe MinIO absent"

spark = (
    SparkSession.builder
    .appName("US06-Bronze-Validation")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.executor.cores", "2")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config(
        "spark.hadoop.fs.s3a.access.key",
        os.environ["MINIO_ROOT_USER"]
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        os.environ["MINIO_ROOT_PASSWORD"]
    )
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark :", spark.version)
print("Master :", sc.master)
print("Application :", sc.appName)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/spark/.ivy2/cache
The jars for the packages stored in: /home/spark/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
io.delta#delta-spark_2.12 added as a dependency
io.graphframes#graphframes-spark3_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e47e7920-f860-4de0-bf04-0f138abffb11;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.7 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.7 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.

Spark : 3.5.7
Master : spark://spark-master:7077
Application : US06-Bronze-Validation


In [2]:
bronze_path = "s3a://lake/bronze/history/transactions_history.jsonl"

print("Source :", bronze_path)

Source : s3a://lake/bronze/history/transactions_history.jsonl


In [3]:
bronze_df = spark.read.text(bronze_path)

bronze_df.printSchema()
bronze_df.show(3, truncate=False)

26/11/07 21:44:45 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- value: string (nullable = true)



+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{"tx_id":"TX-0000000001","type":"P2M","payer_wallet":"W-88001","payee_id":"M-013","device_id":"D-FRAUD-SHARED-000","city":"Marrakech","channel":"APP","status":"COMPLETED","amount_mad":4603.13,"event_time":"2026-11-07T08:00:00Z"} |
|{"tx_id":"TX-0000000002","type":"P2M","payer_wallet":"W-88002","payee_i

In [4]:
expected_count = 50000
actual_count = bronze_df.count()

print("Événements attendus :", expected_count)
print("Événements lus      :", actual_count)

assert actual_count == expected_count, (
    f"Nombre incorrect : {actual_count} au lieu de {expected_count}"
)

print("VALIDATION OK : 50000 événements lus depuis MinIO")

[Stage 1:=======================================>                   (2 + 1) / 3]

Événements attendus : 50000
Événements lus      : 50000
VALIDATION OK : 50000 événements lus depuis MinIO


In [5]:
events_df = spark.read.json(bronze_path)

events_df.printSchema()

events_df.select(
    "tx_id",
    "type",
    "status",
    "amount_mad",
    "event_time"
).show(5, truncate=False)

assert events_df.count() == 50000

assert len(events_df.columns) == 10, (
    f"Schéma inattendu : {events_df.columns}"
)

print("VALIDATION OK : JSON interprété par Spark")

root
 |-- amount_mad: double (nullable = true)
 |-- channel: string (nullable = true)
 |-- city: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- payee_id: string (nullable = true)
 |-- payer_wallet: string (nullable = true)
 |-- status: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- type: string (nullable = true)

+-------------+----+---------+----------+--------------------+
|tx_id        |type|status   |amount_mad|event_time          |
+-------------+----+---------+----------+--------------------+
|TX-0000000001|P2M |COMPLETED|4603.13   |2026-11-07T08:00:00Z|
|TX-0000000002|P2M |COMPLETED|3922.43   |2026-11-07T08:00:00Z|
|TX-0000000003|P2M |COMPLETED|3485.33   |2026-11-07T08:00:00Z|
|TX-0000000004|P2M |COMPLETED|117.81    |2026-11-07T08:00:00Z|
|TX-0000000005|P2M |COMPLETED|804.88    |2026-11-07T08:00:01Z|
+-------------+----+---------+----------+--------------------+
only showing top 5 rows

VALIDAT

In [6]:
spark.stop()

print("SparkSession arrêtée")

SparkSession arrêtée
